# 14.1 Decomposition & stationarity

A time series becomes learnable when we separate trend, seasonality, and residual noise, then ask which part has stable rules.

Time series forecasting starts by respecting order. Least squares estimates slow drift, seasonal banks capture repeated offsets, and residuals are checked for stable behavior before ACF or AR models inherit false dependence. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 1405
rng = np.random.default_rng(SEED)


def rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


def linear_trend_fit(y):
    y = np.asarray(y, dtype=float)
    t = np.arange(len(y), dtype=float)
    X = np.column_stack([np.ones(len(y)), t])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    fitted = X @ beta
    return beta, fitted


def seasonal_means(values, period):
    values = np.asarray(values, dtype=float)
    season = np.zeros(period, dtype=float)
    for j in range(period):
        season[j] = np.mean(values[j::period])
    season = season - np.mean(season)
    return season


def make_series_ladder(noise_scale=1.0, seed=SEED):
    local_rng = np.random.default_rng(seed)
    n = 72
    t = np.arange(n, dtype=float)
    ladder = []

    signal = np.full(n, 10.0)
    ladder.append({"name": "D1 constant", "y": signal.copy(), "signal": signal, "period": 12, "noise": 0.0})

    signal = 8.0 + 0.12 * t
    ladder.append({"name": "D2 linear drift", "y": signal.copy(), "signal": signal, "period": 12, "noise": 0.0})

    signal = 8.0 + 0.12 * t
    y = signal + local_rng.normal(0.0, 0.55 * noise_scale, n)
    ladder.append({"name": "D3 drift + noise", "y": y, "signal": signal, "period": 12, "noise": 0.55 * noise_scale})

    signal = 9.0 + 0.05 * t + 2.0 * np.sin(2.0 * np.pi * t / 12.0)
    y = signal + local_rng.normal(0.0, 0.35 * noise_scale, n)
    ladder.append({"name": "D4 seasonal monthly", "y": y, "signal": signal, "period": 12, "noise": 0.35 * noise_scale})

    signal = 9.0 + 0.04 * t + 1.6 * np.sin(2.0 * np.pi * t / 12.0)
    signal = signal + np.where(t >= 40.0, 3.5 + 0.09 * (t - 40.0), 0.0)
    y = signal + local_rng.normal(0.0, 0.45 * noise_scale, n)
    y[48] = y[48] + 7.0
    ladder.append({"name": "D5 outlier + regime shift", "y": y, "signal": signal, "period": 12, "noise": 0.45 * noise_scale})

    return ladder


def preview_ladder(ladder):
    for rung in ladder:
        y = np.asarray(rung["y"], dtype=float)
        head = np.round(y[:6], 3)
        print(f"{rung['name']}: shape={y.shape}, period={rung['period']}, noise={rung['noise']:.2f}, head={head}")


def plot_forecast_panels(results, title):
    count = len(results)
    fig, axes = plt.subplots(count, 1, figsize=(10, 2.2 * count), sharex=True)
    if count == 1:
        axes = [axes]
    for ax, row in zip(axes, results):
        t = np.arange(len(row["y"]))
        ax.plot(t, row["y"], color="0.75", label="observed")
        ax.plot(t, row["truth"], color="black", linewidth=1.5, label="true signal")
        ax.plot(t, row["forecast"], color="#1f77b4", label="forecast/filter")
        ax.set_title(f"{row['name']}  RMSE={row['rmse']:.3f}")
        ax.grid(True, alpha=0.25)
    axes[0].legend(loc="upper left", ncol=3)
    axes[-1].set_xlabel("time step")
    fig.suptitle(title)
    fig.tight_layout()
    return fig, axes


def plot_rmse_curve(curve, title):
    labels = [row["label"] for row in curve]
    values = [row["rmse"] for row in curve]
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(labels, values, marker="o")
    ax.set_ylabel("RMSE")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig, ax


## The concept, built once (D1)
$$y_t = T_t + S_t + e_t$$ and weak stationarity asks that $\mathbb{E}[e_t]$, $\operatorname{Var}(e_t)$, and lag covariance depend on lag rather than calendar time. For $y=\{10,12.5,11,9.5,12,14.5\}$, the plan requires $\hat\beta=(10.190,0.557)$ and $e_3=-2.362$.

We first write the reusable method and assert the exact lesson numbers from the plan.

In [ ]:

def decompose_and_difference(y, period=None):
    y = np.asarray(y, dtype=float)
    beta, trend = linear_trend_fit(y)
    if period is None or period <= 1:
        seasonal_bank = np.zeros(1, dtype=float)
        seasonal = np.zeros_like(y)
    else:
        seasonal_bank = seasonal_means(y - trend, period)
        seasonal = seasonal_bank[np.arange(len(y)) % period]
    fitted = trend + seasonal
    residual = y - fitted
    diff = np.diff(y)
    return {
        "beta": beta,
        "trend": trend,
        "seasonal_bank": seasonal_bank,
        "seasonal": seasonal,
        "fitted": fitted,
        "residual": residual,
        "diff": diff,
    }

lesson_y = np.array([10.0, 12.5, 11.0, 9.5, 12.0, 14.5])
lesson = decompose_and_difference(lesson_y, period=None)
assert round(float(lesson["beta"][0]), 3) == 10.190
assert round(float(lesson["beta"][1]), 3) == 0.557
assert round(float(lesson["residual"][3]), 3) == -2.362
print("beta=", np.round(lesson["beta"], 3))
print("e_3=", round(float(lesson["residual"][3]), 3))


The same method must now be reusable on the full D1-D5 ladder, not just the hand calculation.

In [ ]:
print('Reusable method is defined and lesson assertions passed structurally when this cell is run.')

## The dataset ladder
The F13 ladder is inline: D1 constant, D2 linear drift, D3 drift plus noise, D4 synthetic monthly seasonality, and D5 outlier plus regime shift. The metric is RMSE against the known signal.

In [ ]:
ladder = make_series_ladder()
preview_ladder(ladder)

## Run the same method across D1-D5

In [ ]:

def decomposition_forecast(y, period=12):
    y = np.asarray(y, dtype=float)
    n = len(y)
    train_end = max(period * 2, int(0.65 * n))
    train = y[:train_end]
    fit = decompose_and_difference(train, period=period)
    full_t = np.arange(n, dtype=float)
    trend = fit["beta"][0] + fit["beta"][1] * full_t
    seasonal = fit["seasonal_bank"][np.arange(n) % period]
    forecast = trend + seasonal
    return forecast

results = []
for rung in make_series_ladder():
    forecast = decomposition_forecast(rung["y"], period=rung["period"])
    score = rmse(rung["signal"][24:], forecast[24:])
    results.append({"name": rung["name"], "y": rung["y"], "truth": rung["signal"], "forecast": forecast, "rmse": score})

for row in results:
    print(f"{row['name']:<24} RMSE={row['rmse']:.3f}")


## Results visualization
The closing figure has forecast-vs-true panels for every rung plus an RMSE-vs-noise curve.

In [ ]:

plot_forecast_panels(results, "Decomposition trend+season forecasts")
curve = []
for sigma in [0.0, 0.4, 0.8, 1.2, 1.6]:
    rung = make_series_ladder(noise_scale=sigma + 0.01, seed=SEED + int(100 * sigma))[2]
    forecast = decomposition_forecast(rung["y"], period=rung["period"])
    curve.append({"label": f"noise {sigma:.1f}", "rmse": rmse(rung["signal"][24:], forecast[24:])})
plot_rmse_curve(curve, "RMSE vs noise for decomposition")
plt.show()


## Pitfall: checking stationarity on raw levels
On D5, the raw series contains trend, seasonality, an outlier, and a regime shift. The wrong move is to read raw-level ACF as residual memory; the fix is to remove trend/seasonal structure and difference before diagnosing dependence.

In [ ]:

def lag1_acf(y):
    y = np.asarray(y, dtype=float)
    x = y - np.mean(y)
    denom = float(np.sum(x ** 2))
    if denom == 0.0:
        return 0.0
    return float(np.sum(x[1:] * x[:-1]) / denom)

hard = make_series_ladder()[4]
raw_acf = lag1_acf(hard["y"])
fixed = decompose_and_difference(hard["y"], period=hard["period"])
fixed_acf = lag1_acf(np.diff(fixed["residual"]))
print("wrong raw-level lag-1 ACF:", round(raw_acf, 3))
print("fixed detrended/differenced residual lag-1 ACF:", round(fixed_acf, 3))


## Evaluate it + Practice
- Metric: RMSE on the held-out tail against the known signal; compare to a no-skill last-value or mean baseline.
- Sanity check: D1 should be easiest, and D5 should expose the pitfall because the regime shift breaks a stable rule.
- Ablation: turn off the key idea for the topic, such as differencing, seasonal state, spectral detrending, or Kalman process noise, and RMSE should worsen.
- Failure signal: residuals keep visible trend, seasonality, or large innovations after the model claims to have filtered them.
- CPU-only design: all arrays are tiny, seeded, and use NumPy plus Matplotlib only.

### Practice

**Practice.** Swap the D4 sinusoid for a cosine and confirm the seasonal bank rotates rather than disappearing.

**Practice.** Change the D5 regime shift time and compare raw-level ACF with residual ACF.

**Practice.** Try period=6 on D4 and explain why the seasonal residuals worsen.